In [1]:
'''Models'''
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from typing import Optional
import pandas as pd
print("import success")

import success


In [ ]:
'''Importing data?'''

In [2]:
'''Model'''
class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)
 
 
class EarningsTransformer(nn.Module):
    """
    Transformer encoder for earnings surprise prediction.
 
    Args:
        n_analysts:    vocabulary size for analyst embedding
        d_emb:         analyst embedding dimension
        d_cont:        number of continuous input features
        d_model:       transformer hidden dimension
        n_heads:       number of attention heads
        n_layers:      number of encoder layers
        d_ff:          feedforward dimension inside encoder
        max_len:       maximum sequence length
        dropout:       dropout rate
        use_analyst_emb: if False, ablates the analyst embedding (research Q1)
    """
    def __init__(
        self,
        n_analysts: int,
        d_emb: int = 16,
        d_cont: int = 4,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        d_ff: int = 128,
        max_len: int = 51,   # +1 for CLS token
        dropout: float = 0.1,
        use_analyst_emb: bool = True,
    ):
        super().__init__()
        self.use_analyst_emb = use_analyst_emb
        self.d_model = d_model
 
        # Analyst embedding (ablation: replace with zeros)
        if use_analyst_emb:
            self.analyst_emb = nn.Embedding(n_analysts, d_emb, padding_idx=0)
            d_input = d_cont + d_emb
        else:
            self.analyst_emb = None
            d_input = d_cont
 
        # Project combined input to d_model
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
        )
 
        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
 
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)
 
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,   # (B, T, D) convention
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
 
        # Output heads
        self.surprise_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
        )
        self.return_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
        )
 
    def forward(
        self,
        cont_features: torch.Tensor,   # (B, T, D_cont)
        analyst_ids: torch.Tensor,      # (B, T)
        pad_mask: torch.Tensor,         # (B, T) True = padding
        return_attention: bool = False,
    ):
        B, T, _ = cont_features.shape
 
        if self.use_analyst_emb:
            emb = self.analyst_emb(analyst_ids)            # (B, T, D_emb)
            x = torch.cat([cont_features, emb], dim=-1)   # (B, T, D_cont+D_emb)
        else:
            x = cont_features                              # (B, T, D_cont)
 
        x = self.input_proj(x)                             # (B, T, D_model)
 
        # Prepend CLS token
        cls = self.cls_token.expand(B, 1, self.d_model)   # (B, 1, D_model)
        x = torch.cat([cls, x], dim=1)                    # (B, T+1, D_model)
 
        # Extend padding mask for CLS (CLS is never padding)
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=pad_mask.device)
        pad_mask_ext = torch.cat([cls_mask, pad_mask], dim=1)  # (B, T+1)
 
        x = self.pos_enc(x)
 
        if return_attention:
            # Manually run layers to collect attention weights
            attn_weights = []
            for layer in self.encoder.layers:
                # attn_output, attn_weight = layer.self_attn(...)
                # PyTorch TransformerEncoderLayer doesn't expose attn weights natively;
                # see AttentionCapture wrapper below for a clean solution.
                x = layer(x, src_key_padding_mask=pad_mask_ext)
            encoded = x
        else:
            encoded = self.encoder(x, src_key_padding_mask=pad_mask_ext)  # (B, T+1, D_model)
 
        cls_out = encoded[:, 0, :]  # (B, D_model)
 
        surprise_pred = self.surprise_head(cls_out).squeeze(-1)   # (B,)
        return_pred   = self.return_head(cls_out).squeeze(-1)      # (B,)
 
        return surprise_pred, return_pred
 
 
# ─── Attention Capture Wrapper (for research analysis) ───────
 
class AttentionCapture(nn.Module):
    """Wraps a single TransformerEncoderLayer to expose attention weights."""
    def __init__(self, layer: nn.TransformerEncoderLayer):
        super().__init__()
        self.layer = layer
        self._attn_weights = None
 
    @property
    def attn_weights(self):
        return self._attn_weights
 
    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        # Re-implement the self-attention block to get weights
        src2, attn = self.layer.self_attn(
            src, src, src,
            attn_mask=src_mask,
            key_padding_mask=src_key_padding_mask,
            need_weights=True,
            average_attn_weights=True,
        )
        self._attn_weights = attn.detach().cpu()  # (B, T, T)
        # Continue with the rest of the encoder layer manually
        src = src + self.layer.dropout1(src2)
        src = self.layer.norm1(src)
        src2 = self.layer.linear2(self.layer.dropout(self.layer.activation(self.layer.linear1(src))))
        src = src + self.layer.dropout2(src2)
        src = self.layer.norm2(src)
        return src

In [ ]:
'''Baselines'''
class ZeroBaseline:
    """Always predicts surprise = 0."""
    def predict(self, X):
        return np.zeros(len(X))
 
 
class ConsensusBaseline:
    """Predicts using the most recent consensus (last estimate_value in sequence)."""
    def predict(self, last_consensus: np.ndarray):
        # last_consensus: (N,) — the most recent estimate_value per announcement
        return last_consensus
 
 
def build_linear_features(df: pd.DataFrame) -> np.ndarray:
    """
    Construct summary statistic features per announcement for linear regression.
    Returns array of shape (N_announcements, n_features).
    """
    rows = []
    for _, grp in df.groupby("announcement_id", sort=False):
        grp = grp.sort_values("days_to_earnings", ascending=False)
        mags = grp["revision_magnitude"].values
        rows.append({
            "n_revisions":       len(grp),
            "mean_magnitude":    mags.mean(),
            "std_magnitude":     mags.std() if len(grp) > 1 else 0.0,
            "min_magnitude":     mags.min(),
            "max_magnitude":     mags.max(),
            "last_estimate":     grp["estimate_value"].iloc[-1],
            "first_estimate":    grp["estimate_value"].iloc[0],
            "total_revision":    grp["estimate_value"].iloc[-1] - grp["estimate_value"].iloc[0],
            "days_since_last":   grp["days_to_earnings"].iloc[-1],
            "frac_up":           (mags > 0).mean(),
            "recency_wtd_mean":  np.average(
                                     grp["estimate_value"].values,
                                     weights=1.0 / (grp["days_to_earnings"].values + 1)
                                 ),
        })
    return pd.DataFrame(rows).values.astype(np.float32)
 
 
class LinearBaseline:
    def __init__(self, alpha: float = 1.0):
        self.scaler = StandardScaler()
        self.model = Ridge(alpha=alpha)
 
    def fit(self, X: np.ndarray, y: np.ndarray):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
 
    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(self.scaler.transform(X))

In [3]:
'''Training loop'''
def train_epoch(
    model: EarningsTransformer,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    lambda_return: float = 0.5,
) -> float:
    model.train()
    total_loss = 0.0
    mse = nn.MSELoss()
 
    for batch in loader:
        cont    = batch["cont_features"].to(device)
        aids    = batch["analyst_ids"].to(device)
        mask    = batch["pad_mask"].to(device)
        surp    = batch["surprise"].to(device)
        ret     = batch["ret"].to(device)
        ret_ok  = batch["ret_valid"].to(device)
 
        optimizer.zero_grad()
        surp_pred, ret_pred = model(cont, aids, mask)
 
        loss = mse(surp_pred, surp)
 
        # Add return loss only where labels are valid
        if ret_ok.any() and lambda_return > 0:
            loss = loss + lambda_return * mse(ret_pred[ret_ok], ret[ret_ok])
 
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
 
    return total_loss / len(loader)
 
 
@torch.no_grad()
def evaluate(
    model: EarningsTransformer,
    loader: DataLoader,
    device: torch.device,
) -> dict:
    model.eval()
    all_preds, all_targets = [], []
 
    for batch in loader:
        cont  = batch["cont_features"].to(device)
        aids  = batch["analyst_ids"].to(device)
        mask  = batch["pad_mask"].to(device)
        surp  = batch["surprise"].numpy()
 
        surp_pred, _ = model(cont, aids, mask)
        all_preds.append(surp_pred.cpu().numpy())
        all_targets.append(surp)
 
    preds   = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
 
    mse_val = np.mean((preds - targets) ** 2)
    mae_val = np.mean(np.abs(preds - targets))
    dir_acc = np.mean(np.sign(preds) == np.sign(targets))
 
    return {"mse": mse_val, "mae": mae_val, "dir_acc": dir_acc}
 
 
def train(
    model: EarningsTransformer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_epochs: int = 50,
    lr: float = 1e-4,
    weight_decay: float = 1e-2,
    lambda_return: float = 0.5,
    patience: int = 10,
    device: Optional[torch.device] = None,
) -> EarningsTransformer:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
 
    total_steps = n_epochs * len(train_loader)
    warmup_steps = int(0.05 * total_steps)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps, pct_start=warmup_steps / total_steps
    )
 
    best_val_mse = float("inf")
    best_state = None
    no_improve = 0
 
    for epoch in range(1, n_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, device, lambda_return)
        scheduler.step()
        val_metrics = evaluate(model, val_loader, device)
 
        print(
            f"Epoch {epoch:03d} | train_loss={train_loss:.4f} | "
            f"val_mse={val_metrics['mse']:.4f} | val_mae={val_metrics['mae']:.4f} | "
            f"val_dir_acc={val_metrics['dir_acc']:.3f}"
        )
 
        if val_metrics["mse"] < best_val_mse:
            best_val_mse = val_metrics["mse"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break
 
    model.load_state_dict(best_state)
    return model